# Implementación — Robustez de Detectores de Phishing ante Correos Generados por LLM

**Curso:** CC3094 - Security Data Science  
**Proyecto:** Robustez de modelos de detección de phishing entrenados con correos humanos frente a correos generados por modelos de lenguaje grandes
**Miembros:** Edwin Ortega - 22305 y Esteban Zambrano 22119

### Objetivo de esta fase
En esta segunda fase se implementan y evalúan modelos de clasificación para analizar si un detector de phishing entrenado únicamente con correos escritos por humanos mantiene su desempeño cuando se enfrenta a correos generados por LLMs.

El enfoque principal del experimento consiste en entrenar los modelos con correos humanos y evaluarlos en dos escenarios:

1. Correos humanos, como evaluación in-distribution.
2. Correos generados por LLM, como evaluación bajo cambio de distribución.

Esta fase responde al entregable de implementación, refinamiento y evaluación de modelos mediante métricas como accuracy, precision, recall, F1-score y curva ROC.

### Enfoque del proyecto
Aunque el dataset incluye correos humanos y correos generados por LLM, el objetivo final del proyecto es evaluar la robustez de modelos entrenados con correos humanos cuando se enfrentan a correos generados por LLM, sin reentrenamiento.

In [1]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score
)

import warnings
warnings.filterwarnings("ignore")

In [5]:
# Load processed dataset

df = pd.read_csv("../data/processed/emails_processed.csv")

df.head()


,text,label_binary,source_type,email_group,sender,receiver,date,subject,body,urls,...,char_count,word_count,avg_word_length,url_count,exclamation_count,question_count,uppercase_count,uppercase_ratio,digit_count,urgent_keyword_count
0,Starting IC with wizard Hi I am running the IR...,0,human,legit,Jesus Miguel Recuenco Ezquerra <JMRECU@telelin...,handy board <handyboard@media.mit.edu>,2019-10-29 22:53:50,Starting IC with wizard,Hi\r\n\r\n\t\tI am running the IR test program...,0,...,211,44,3.818182,0,0,1,18,0.109756,0,0
1,Trade Me -- A question on your auction: Auctio...,0,human,legit,Trade Me <xfnbqb@trademe.co.nz>,user2.4@gvc.ceas-challenge.cc,2008-08-06 13:53:26,Trade Me -- A question on your auction: Auctio...,\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r...,0,...,739,127,4.826772,2,1,0,26,0.046181,25,3
2,Trade Me - A request from a Trade Me member. A...,0,human,legit,Trade Me <xfnbqb@trademe.co.nz>,user2.4@gvc.ceas-challenge.cc,2008-08-06 13:45:53,Trade Me - A request from a Trade Me member. A...,\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\nTrade ...,0,...,358,61,4.885246,0,0,0,27,0.102662,22,2
3,RE: NorthTec Account/Password Hi Tony Not sure...,0,human,legit,Kevin Jacobson <wffjeanja@northtec.ac.nz>,user2.1@gvc.ceas-challenge.cc,2008-08-06 13:43:27,RE: NorthTec Account/Password,"Hi Tony\r\nNot sure why it didn't work, but I ...",1,...,3180,562,4.660142,4,0,2,175,0.078829,135,19
4,2008 timetable Kindly suggest changes --------...,0,human,legit,Albert van Aardt <zfdrqfguo@northtec.ac.nz>,user2.1@gvc.ceas-challenge.cc,2008-08-06 13:26:57,2008 timetable,Kindly suggest changes\r\n\r\n----------------...,0,...,206,24,7.625000,0,0,0,14,0.121739,15,0


In [6]:
# Initial dataset inspection

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nSource type distribution:")
print(df["source_type"].value_counts())

print("\nEmail group distribution:")
print(df["email_group"].value_counts())

print("\nSource type vs email group:")
print(pd.crosstab(df["source_type"], df["email_group"]))

print("\nBinary label distribution:")
print(df["label_binary"].value_counts())

Dataset shape: (3200, 21)

Columns:
Index(['text', 'label_binary', 'source_type', 'email_group', 'sender',
       'receiver', 'date', 'subject', 'body', 'urls', 'label', 'char_count',
       'word_count', 'avg_word_length', 'url_count', 'exclamation_count',
       'question_count', 'uppercase_count', 'uppercase_ratio', 'digit_count',
       'urgent_keyword_count'],
      dtype='object')

Source type distribution:
source_type
llm      1998
human    1202
Name: count, dtype: int64

Email group distribution:
email_group
legit       1700
phishing    1500
Name: count, dtype: int64

Source type vs email group:
email_group  legit  phishing
source_type                 
human          702       500
llm            998      1000

Binary label distribution:
label_binary
0    1700
1    1500
Name: count, dtype: int64


In [7]:
pd.crosstab(
    df["source_type"],
    df["email_group"],
    margins=True
)

email_group,legit,phishing,All
source_type,,,
human,702,500,1202
llm,998,1000,1998
All,1700,1500,3200


## Descripción del dataset

Como parte de la Fase 1 del proyecto se realizó el análisis exploratorio e ingeniería de características sobre el conjunto de datos utilizado en esta implementación. En esta segunda fase se reutiliza el dataset procesado generado previamente (`emails_processed.csv`), el cual contiene correos legítimos y correos phishing provenientes de dos orígenes:

- Correos escritos por humanos (`source_type = human`)
- Correos generados por modelos de lenguaje (`source_type = llm`)

Para responder la pregunta de investigación, los datos humanos serán utilizados para entrenamiento del modelo, mientras que los correos generados por LLM serán reservados exclusivamente para evaluación bajo cambio de distribución (*distribution shift*).

La variable objetivo utilizada para clasificación es:

- `label_binary = 0` → correo legítimo  
- `label_binary = 1` → correo phishing

En esta primera parte se implementará un detector basado en contenido textual usando TF-IDF y Support Vector Machines.